In [5]:
import pandas as pd
import json
from pathlib import Path

# ============================================================
# 1. GABUNG 3 FILE CSV
# ============================================================

candidates = [
    Path.cwd(),
    Path.cwd() / 'final_merge_data',
    Path.cwd() / 'data',
    Path.cwd() / 'final_merge_data' / 'data',
    Path(r'c:\kerjaan_magang\final_merge_data\data'),
    Path(r'c:\kerjaan_magang\data')
]

base_dir = next(
    (
        p for p in candidates
        if p.exists() and any((p / f).exists() for f in [
            'cleaned_cleanser1.csv',
            'cleaned_cleanser2.csv',
            'cleaned_cleanser3.csv'
        ])
    ),
    Path.cwd() / 'final_merge_data' / 'data'
)

files = [
    'cleaned_cleanser1.csv',
    'cleaned_cleanser2.csv',
    'cleaned_cleanser3.csv'
]

dfs = []

for filename in files:
    file_path = base_dir / filename

    if not file_path.exists():
        print(f'⚠️ File tidak ditemukan: {file_path}')
        continue

    try:
        with open(file_path, 'r', encoding='utf-8-sig', errors='replace') as f:
            sample = f.read(4096)

        delimiter = ';' if sample and ';' in sample.splitlines()[0] else ','
        df = pd.read_csv(file_path, sep=delimiter, encoding='utf-8-sig')
        dfs.append(df)
        print(f'✅ Dibaca: {file_path.name} ({len(df):,} baris)')

    except Exception as e:
        print(f'⚠️ Gagal membaca {file_path.name}: {e}')

if not dfs:
    raise FileNotFoundError(
        f"Tidak ada file CSV yang berhasil dibaca di folder: {base_dir}"
    )

final = pd.concat(dfs, ignore_index=True)
final.columns = [str(col).strip() for col in final.columns]

print(f'\nTotal data sebelum dedup: {len(final):,}')


# ============================================================
# 2. KONVERSI KOORDINAT KE NUMERIC
# ============================================================

final['Latitude'] = pd.to_numeric(
    final['Latitude'],
    errors='coerce'
)

final['Longitude'] = pd.to_numeric(
    final['Longitude'],
    errors='coerce'
)


# ============================================================
# 3. HAPUS DUPLIKAT
# ============================================================

final = final.drop_duplicates(
    subset=[
        'Nama SPPG',
        'Latitude',
        'Longitude'
    ],
    keep='first'
).reset_index(drop=True)

print(f'Total data setelah dedup: {len(final):,}')


# ============================================================
# 4. HAPUS KOORDINAT TIDAK VALID
# ============================================================

final = final[
    final['Latitude'].between(-90, 90, inclusive='both') &
    final['Longitude'].between(-180, 180, inclusive='both')
].reset_index(drop=True)

print(f'Total data dengan koordinat valid: {len(final):,}')


# ============================================================
# 5. RESET NOMOR SPPG
# ============================================================

final['No SPPG'] = range(1, len(final) + 1)


# ============================================================
# 6. EXPORT CSV
# ============================================================

csv_output = 'master_cleaned_sppg_final.csv'

final.to_csv(
    csv_output,
    index=False,
    encoding='utf-8-sig'
)

print(f'\n✅ CSV tersimpan: {csv_output}')
print(f'   Total titik SPPG: {len(final):,}')


# ============================================================
# 7. CONVERT KE GEOJSON
# ============================================================

features = []

for _, row in final.iterrows():

    # Skip jika koordinat kosong
    if pd.isna(row['Latitude']) or pd.isna(row['Longitude']):
        continue

    feature = {
        "type": "Feature",

        "geometry": {
            "type": "Point",
            "coordinates": [
                float(row["Longitude"]),
                float(row["Latitude"])
            ]
        },

        "properties": {
            "No SPPG": str(row["No SPPG"]),
            "Nama SPPG": str(row["Nama SPPG"]),
            "Provinsi SPPG": str(row.get("Provinsi SPPG", "")),
            "Kabupaten/Kota SPPG": str(
                row.get("Kabupaten/Kota SPPG", "")
            ),
            "Kecamatan SPPG": str(
                row.get("Kecamatan SPPG", "")
            ),
            "Alamat SPPG": str(
                row.get("Alamat SPPG", "")
            ),
            "Latitude": float(row["Latitude"]),
            "Longitude": float(row["Longitude"]),
            "link": str(row.get("link", ""))
        }
    }

    features.append(feature)


# ============================================================
# 8. SIMPAN GEOJSON
# ============================================================

geojson_output = 'master_cleaned_sppg_final.geojson'

with open(
    geojson_output,
    'w',
    encoding='utf-8'
) as f:

    json.dump(
        {
            "type": "FeatureCollection",
            "features": features
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print(f'✅ GeoJSON tersimpan: {geojson_output}')
print(f'   Total titik GeoJSON: {len(features):,}')

✅ Dibaca: cleaned_cleanser1.csv (6,234 baris)
✅ Dibaca: cleaned_cleanser2.csv (7,106 baris)
✅ Dibaca: cleaned_cleanser3.csv (7,029 baris)

Total data sebelum dedup: 20,369
Total data setelah dedup: 20,351
Total data dengan koordinat valid: 20,349

✅ CSV tersimpan: master_cleaned_sppg_final.csv
   Total titik SPPG: 20,349
✅ GeoJSON tersimpan: master_cleaned_sppg_final.geojson
   Total titik GeoJSON: 20,349
